# 10 · Regularization, Generalization and Learning Theory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/10-regularization-generalization/10_regularization_generalization.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-10-regularization-generalization.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 10](https://github.com/anuxlab/deep-learning-systems/wiki/Module-10) |
| **Level** | Intermediate |
| **Typical CPU runtime** | ~1–2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 06, 07 |

## What you will do

- Detect data leakage with a demonstration that gives 'amazing' accuracy on random labels.
- Measure bias–variance empirically and see over/underfitting in train–validation curves.
- Compare L1 vs L2 and show L2 penalty ≡ weight decay in SGD.
- Implement inverted dropout, label smoothing, input noise, mixup; use early stopping and ensembling.
- Reproduce double descent with min-norm random-feature regression.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · Data leakage: the most expensive bug
Fit *anything* on the whole dataset before splitting and information leaks from test to train. Classic demo: **random labels** (so no model can beat 50%) with 5,000 noise features; pick the 20 "best" features using *all* labels, then cross-validate. The leaky pipeline looks brilliant; the correct pipeline (selection *inside* each fold) reports ~50%.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
r = np.random.default_rng(0); Xn = r.normal(size=(100, 5000)); yn = r.integers(0, 2, 100)
Xsel = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)                             # LEAK: selection sees all labels
leaky = cross_val_score(LogisticRegression(max_iter=1000), Xsel, yn, cv=5).mean()
proper = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)), Xn, yn, cv=5).mean()
print(f"leaky CV accuracy on RANDOM labels: {leaky:.2f}   |   correct pipeline: {proper:.2f}")
check(leaky > .75 and proper < .65, "leakage creates fake accuracy; a Pipeline removes it")
METRICS.update(leaky_acc=leaky, proper_acc=proper)

## 2 · Under/overfitting and the bias–variance trade-off
For squared error: $\mathbb E[(y-\hat f)^2]=\underbrace{(\mathbb E\hat f-f)^2}_{\text{bias}^2}+\underbrace{\mathrm{Var}(\hat f)}_{\text{variance}}+\sigma^2$. We fit polynomials of increasing degree to many independent training sets drawn from $y=\sin(2\pi x)+\epsilon$ and measure each term at fixed test points.

In [ ]:
f_true = lambda x: np.sin(2 * np.pi * x); sigma = .3; xt = np.linspace(.02, .98, 60); degrees = range(0, 13); reps = 300; n = 25
bias2, var, tr_err, te_err = [], [], [], []
for d in degrees:
    preds, trs = [], []
    for s in range(reps):
        r = np.random.default_rng(s); x = r.uniform(0, 1, n); y = f_true(x) + sigma * r.normal(size=n)
        with np.errstate(all="ignore"): c = np.polyfit(x, y, d)
        preds.append(np.polyval(c, xt)); trs.append(np.mean((np.polyval(c, x) - y) ** 2))
    P = np.array(preds); bias2.append(np.mean((P.mean(0) - f_true(xt)) ** 2)); var.append(P.var(0).mean()); tr_err.append(np.mean(trs))
    te_err.append(bias2[-1] + var[-1] + sigma ** 2)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(degrees, bias2, "o-", label="bias²"); ax[0].plot(degrees, var, "s-", label="variance"); ax[0].plot(degrees, te_err, "k-", label="expected test error"); ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_xlabel("model complexity (degree)")
ax[1].plot(degrees, tr_err, "o-", label="train error"); ax[1].plot(degrees, te_err, "s-", label="test error"); ax[1].set_ylim(0, .6); ax[1].legend(); ax[1].set_xlabel("degree"); ax[1].set_title("Generalisation gap opens with complexity")
show("bias_variance")
best = int(np.argmin(te_err)); print("best degree:", best, " test err:", round(te_err[best], 4))
check(bias2[0] > bias2[best] and var[-1] > var[best], "low degree: high bias; high degree: high variance")
check(2 <= best <= 8, "optimal complexity is in the middle (U-shaped test error)")
check(tr_err[-1] < tr_err[0], "training error decreases monotonically-ish with complexity")

## 3 · L2, L1 and weight decay
- **L2 (ridge):** $\min\|y-Xw\|^2+\lambda\|w\|^2$ → shrinks all weights; closed form $w=(X^\top X+\lambda I)^{-1}X^\top y$.
- **L1 (lasso):** $+\lambda\|w\|_1$ → drives many weights to exactly **zero** (sparsity) because the penalty has a corner at 0.
- **Weight decay** in SGD: $w\leftarrow(1-\eta\lambda)w-\eta g$. For plain SGD this is *identical* to adding $\frac\lambda2\|w\|^2$ to the loss (we verify). For Adam they differ — hence AdamW.

In [ ]:
from sklearn.linear_model import Lasso, Ridge, LinearRegression
r = np.random.default_rng(0); n, d = 80, 40; wtrue = np.zeros(d); wtrue[:5] = [3, -2, 1.5, 4, -3]
X = r.normal(size=(n, d)); y = X @ wtrue + r.normal(size=n)
ols, rid, las = LinearRegression().fit(X, y), Ridge(alpha=30).fit(X, y), Lasso(alpha=.15).fit(X, y)
print(f"non-zero coefficients: OLS {np.sum(np.abs(ols.coef_)>1e-8)}, ridge {np.sum(np.abs(rid.coef_)>1e-8)}, lasso {np.sum(np.abs(las.coef_)>1e-8)}  (truth 5)")
fig, ax = plt.subplots(figsize=(10, 3.6)); w_ = .27
for i, (nme, m) in enumerate([("OLS", ols), ("Ridge", rid), ("Lasso", las)]): ax.bar(np.arange(d) + (i - 1) * w_, m.coef_, w_, label=nme)
ax.plot(np.arange(d), wtrue, "k*", label="truth"); ax.legend(); ax.set_title("Coefficient estimates"); show("l1_vs_l2")
check(np.sum(np.abs(las.coef_) < 1e-8) > 20 and np.sum(np.abs(rid.coef_) < 1e-8) == 0, "lasso is sparse; ridge shrinks but never zeroes")
Xte = r.normal(size=(2000, d)); yte = Xte @ wtrue
print("test MSE  OLS %.3f  ridge %.3f  lasso %.3f" % tuple(np.mean((m.predict(Xte) - yte) ** 2) for m in (ols, rid, las)))

# L2-in-loss == weight decay for SGD
w0 = r.normal(size=d); lam, lr = .1, .01
w_a, w_b = w0.copy(), w0.copy()
for _ in range(50):
    g = X.T @ (X @ w_a - y) / n; w_a = w_a - lr * (g + lam * w_a)               # penalty gradient inside g
    g = X.T @ (X @ w_b - y) / n; w_b = (1 - lr * lam) * w_b - lr * g           # decoupled decay
check(np.allclose(w_a, w_b), "SGD: L2 penalty ≡ weight decay")

## 4 · Dropout, label smoothing, noise, mixup, early stopping, ensembles
**Inverted dropout:** zero each unit with probability $p$ and scale survivors by $1/(1-p)$ so $\mathbb E[\text{out}]=\text{in}$ — then nothing changes at test time. We verify the expectation and compare with `nn.Dropout`.

In [ ]:
def dropout(x, p, rng): mask = (rng.random(x.shape) >= p) / (1 - p); return x * mask
x = np.ones((2000, 100)); rng = np.random.default_rng(0)
for p in [.1, .5, .9]:
    o = dropout(x, p, rng); print(f"p={p}: mean output {o.mean():.3f} (should be 1), zeros {np.mean(o==0):.3f}")
    check(abs(o.mean() - 1) < .02 and abs(np.mean(o == 0) - p) < .01, f"inverted dropout p={p}: unbiased, correct drop rate")
d = nn.Dropout(.5); d.train(); a = d(torch.ones(10000)); d.eval(); b = d(torch.ones(5))
check(abs(a.mean().item() - 1) < .05 and torch.all(b == 1), "torch Dropout: scaled in train mode, identity in eval mode")

### An overfitting experiment on digits (small training set)
Only 200 training samples and a large MLP → overfits quickly. We compare: baseline, weight decay, dropout, label smoothing, input noise, mixup — all tracked with validation accuracy. Then **early stopping** (restore the best-validation epoch) and a 5-model **ensemble**.

In [ ]:
from sklearn.datasets import load_digits
D = load_digits(); Xd = torch.tensor(D.data / 16., dtype=torch.float32); yd = torch.tensor(D.target)
idx = np.random.default_rng(0).permutation(len(Xd)); tr, va, te = idx[:200], idx[200:800], idx[800:]
def run(method, seed=0, epochs=150, ret_model=False):
    torch.manual_seed(seed); p_drop = .5 if method == "dropout" else 0.
    m = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(256, 10))
    o = torch.optim.SGD(m.parameters(), lr=.05, momentum=.9, weight_decay=5e-3 if method == "weight decay" else 0.)
    X, y = Xd[tr], yd[tr]; hist = []; best = (-1, None, 0)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(X))
        for s in range(0, len(X), 20):
            b = perm[s:s + 20]; xb, yb = X[b], y[b]
            if method == "noise": xb = xb + .3 * torch.randn_like(xb)
            if method == "mixup":
                lam = np.random.beta(.4, .4); j = torch.randperm(len(xb)); xb = lam * xb + (1 - lam) * xb[j]
                out = m(xb); loss = lam * F.cross_entropy(out, yb) + (1 - lam) * F.cross_entropy(out, yb[j])
            else: loss = F.cross_entropy(m(xb), yb, label_smoothing=.1 if method == "label smoothing" else 0.)
            o.zero_grad(); loss.backward(); o.step()
        m.eval()
        with torch.no_grad(): tl = F.cross_entropy(m(X), y).item(); vl = F.cross_entropy(m(Xd[va]), yd[va]).item(); vacc = (m(Xd[va]).argmax(1) == yd[va]).float().mean().item()
        hist.append((tl, vl, vacc))
        if vacc > best[0]: best = (vacc, {k: v.clone() for k, v in m.state_dict().items()}, ep)
    with torch.no_grad(): tacc = (m(Xd[te]).argmax(1) == yd[te]).float().mean().item()
    return (hist, tacc, best, m) if ret_model else (hist, tacc, best)
methods = ["baseline", "weight decay", "dropout", "label smoothing", "noise", "mixup"]; R = {}
with timer("regularisation_sweep"):
    for mth in methods: R[mth] = run(mth)
for mth, (h, tacc, best) in R.items(): print(f"{mth:16s} final val acc {h[-1][2]:.3f}  best val acc {best[0]:.3f} @epoch {best[2]:3d}  test acc {tacc:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for mth in ["baseline", "weight decay", "dropout"]:
    h = np.array(R[mth][0]); ax[0].plot(h[:, 1], label=f"{mth} val loss")
ax[0].set_ylim(0, 1.5); ax[0].legend(); ax[0].set_xlabel("epoch"); ax[0].set_title("Validation loss (overfitting = rising)")
ax[1].bar(methods, [R[m][1] for m in methods]); ax[1].set_ylim(.8, 1); ax[1].set_title("Test accuracy"); plt.setp(ax[1].get_xticklabels(), rotation=30)
show("regularization_methods")
check(max(R[m][1] for m in methods[1:]) >= R["baseline"][1], "at least one regulariser matches or beats the baseline")
METRICS["reg_test_acc"] = {m: R[m][1] for m in methods}

In [ ]:
# Early stopping + ensembling (averaging softmax outputs of 5 seeds)
hist, tacc, best, _ = run("baseline", ret_model=True)
print(f"baseline: stopped at best epoch {best[2]} -> val acc {best[0]:.3f}")
models = []
for sd in range(5):
    h, ta, b, m = run("dropout", seed=sd, ret_model=True); models.append(m)
with torch.no_grad():
    probs = torch.stack([F.softmax(m(Xd[te]), 1) for m in models]); singles = [(p.argmax(1) == yd[te]).float().mean().item() for p in probs]; ens = (probs.mean(0).argmax(1) == yd[te]).float().mean().item()
print("single models:", np.round(singles, 3), " ensemble:", round(ens, 3))
check(ens >= np.mean(singles), "ensemble accuracy ≥ average single-model accuracy")
METRICS["ensemble_acc"] = ens

## 5 · Double descent
Classical theory says test error is U-shaped in model size. Modern over-parameterised models show a **second descent** beyond the *interpolation threshold* (number of features $p\approx n$ training points). Using **random ReLU features** with the **minimum-norm** least-squares solution (`np.linalg.pinv`), the test error peaks near $p=n$ and falls again as $p\gg n$.

In [ ]:
n_train, d_in = 40, 5; r = np.random.default_rng(0); wstar = r.normal(size=d_in)
def data(n, noise): X = r.normal(size=(n, d_in)); return X, np.tanh(X @ wstar) + noise * r.normal(size=n)
Xtr_, ytr_ = data(n_train, .3); Xte_, yte_ = data(2000, 0.0)
ps = np.unique(np.r_[np.arange(2, 40, 3), np.arange(36, 46), np.geomspace(46, 800, 14).astype(int)]); errs = []
for p in ps:
    e = []
    for rep in range(15):
        rr = np.random.default_rng(100 + rep); Wf = rr.normal(size=(d_in, p)) / np.sqrt(d_in)
        phi = lambda X: np.maximum(0, X @ Wf)
        coef = np.linalg.pinv(phi(Xtr_)) @ ytr_; e.append(np.mean((phi(Xte_) @ coef - yte_) ** 2))
    errs.append(np.median(e))
plt.semilogy(ps, errs, "o-"); plt.axvline(n_train, color="r", ls="--", label=f"interpolation threshold p=n={n_train}"); plt.xlabel("number of random features p"); plt.ylabel("test MSE (median of 15)"); plt.legend(); plt.title("Double descent"); show("double_descent")
errs = np.array(errs); peak = ps[np.argmax(errs)]; print("peak test error at p =", peak)
check(abs(peak - n_train) <= 6, "test error peaks at the interpolation threshold")
check(errs[-1] < errs[np.argmax(errs)] / 10, "error falls again far into the over-parameterised regime")
METRICS["double_descent_peak_p"] = int(peak)

In [ ]:
save_metrics("10", METRICS)
log.info('notebook 10 finished OK')

## Wrap-up

## Exercises (do these next)

1. Add data augmentation (random shifts of the 8×8 images) and compare to the other regularisers.
2. Sweep dropout p and plot validation accuracy; where is the optimum?
3. Show that adding noise to the double-descent labels makes the peak taller; add ridge regularisation and watch the peak disappear.

## Where to read more

- Wiki chapter: [Module 10](https://github.com/anuxlab/deep-learning-systems/wiki/Module-10) (derivations, worked example, Q&A).
- Hastie, Tibshirani & Friedman, *ESL* ch. 3, 7; Srivastava et al. (2014) Dropout; Zhang et al. (2017) *Rethinking generalization*; Belkin et al. (2019) double descent; Zhang et al. (2018) mixup.